# Automated experiments: 10 runs × {sequential, alternating} × {gaussian, gamma} + paired t-test

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from scipy.stats import ttest_rel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

Using device: cuda


## 1. experiment settings

In [2]:
QUICK_TEST = False          # True -> tiny dry run to verify the pipeline; False -> real 10-run experiment

n_runs            = 10
base_seed         = 0
xlsx_path         = 'results/experiment_results.xlsx'
save_models       = True             # save trained .pth models
models_dir        = 'saved_models'   # all .pth files go in this single folder

timesteps         = 1000
theta0            = 0.001
gen_steps         = 50       # DDIM steps for ALL generation (fast)
alpha_coop        = 0.1      # shared cooperative-loss weight (alternating)
batch_size        = 64
learning_rate     = 2e-4
adam_betas        = (0.5, 0.9)

seq_diffusion_epochs  = 50
seq_classifier_epochs = 50
alt_warmup_epochs = 10
alt_epochs        = 40
eval_every        = 2
baseline_epochs   = 50

num_classes       = 3
image_size        = 28
target_per_class  = 800
per_class_test    = 500

if QUICK_TEST:
    n_runs = 2
    seq_diffusion_epochs = seq_classifier_epochs = 1
    alt_warmup_epochs = 1; alt_epochs = 2; eval_every = 1; baseline_epochs = 1
    gen_steps = 5; timesteps = 50; target_per_class = 120; per_class_test = 100
    xlsx_path = 'results/experiment_results_QUICKTEST.xlsx'; models_dir = 'saved_models_QUICKTEST'
    print('QUICK_TEST mode: tiny settings, results are meaningless (pipeline check only).')

## 2. data

In [3]:
SEL    = [0, 2, 3]                 # Fashion-MNIST: 0=T-shirt/top, 2=Pullover, 3=Dress
REMAP  = {0: 0, 2: 1, 3: 2}
NAMES  = ['T-shirt', 'Pullover', 'Dress']
train_counts = {0: 800, 2: 80, 3: 80}    # ratio 0.10

data_root = os.path.join(os.path.dirname(os.getcwd()), 'data')
fm_train = datasets.FashionMNIST(root=data_root, train=True,  download=True)
fm_test  = datasets.FashionMNIST(root=data_root, train=False, download=True)

def _take(ds, orig, idx):
    imgs = (ds.data[idx].float() / 255.0); imgs = (imgs - 0.5) / 0.5
    return imgs.unsqueeze(1), torch.full((len(idx),), REMAP[orig], dtype=torch.long)

def build_train(seed):
    """Resample the imbalanced training subset for this run (per-seed)."""
    rng = np.random.RandomState(seed)
    X, Y = [], []
    for orig in SEL:
        pool = (fm_train.targets == orig).nonzero(as_tuple=False).squeeze(1).numpy()
        idx = torch.as_tensor(rng.choice(pool, train_counts[orig], replace=False))
        x, y = _take(fm_train, orig, idx); X.append(x); Y.append(y)
    return torch.cat(X), torch.cat(Y)

Xte, Yte = [], []
for orig in SEL:
    idx = (fm_test.targets == orig).nonzero(as_tuple=False).squeeze(1)[:per_class_test]
    x, y = _take(fm_test, orig, idx); Xte.append(x); Yte.append(y)
X_test, y_test = torch.cat(Xte), torch.cat(Yte)
print(f"test set: {len(y_test)} samples ({per_class_test}/class)")

test set: 1500 samples (500/class)


## 3. models (conditional U-Net + unified Diffusion + classifier)

In [4]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, time):
        half = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half - 1)
        emb = torch.exp(torch.arange(half, device=time.device) * -emb)
        emb = time.float()[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class Block(nn.Module):
    def __init__(self, i, o, td):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(td, o))
        self.c1 = nn.Conv2d(i, o, 3, padding=1); self.c2 = nn.Conv2d(o, o, 3, padding=1)
        self.norm = nn.GroupNorm(8, o)
    def forward(self, x, t):
        h = self.c1(x) + self.time_mlp(t)[:, :, None, None]
        h = self.norm(F.silu(h)); h = self.c2(h)
        return self.norm(F.silu(h))

class UNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__(); td = 32
        self.time_mlp = nn.Sequential(SinusoidalPositionEmbeddings(td),
                                      nn.Linear(td, td), nn.SiLU(), nn.Linear(td, td))
        self.label_emb = nn.Embedding(num_classes, td)
        self.d1 = Block(1, 64, td); self.d2 = Block(64, 128, td)
        self.d3 = Block(128, 256, td); self.d4 = Block(256, 512, td)
        self.bot = Block(512, 512, td)
        self.u1 = Block(1024, 512, td); self.u2 = Block(768, 256, td)
        self.u3 = Block(384, 128, td); self.u4 = Block(192, 64, td)
        self.out = nn.Conv2d(64, 1, 1)
    def forward(self, x, t, y):
        t = self.time_mlp(t) + self.label_emb(y)
        h1 = self.d1(x, t); h2 = self.d2(F.avg_pool2d(h1, 2), t)
        h3 = self.d3(F.avg_pool2d(h2, 2), t); h4 = self.d4(F.avg_pool2d(h3, 2), t)
        h = self.bot(F.avg_pool2d(h4, 2), t)
        h = F.interpolate(h, size=h4.shape[2:], mode='nearest'); h = self.u1(torch.cat([h, h4], 1), t)
        h = F.interpolate(h, size=h3.shape[2:], mode='nearest'); h = self.u2(torch.cat([h, h3], 1), t)
        h = F.interpolate(h, size=h2.shape[2:], mode='nearest'); h = self.u3(torch.cat([h, h2], 1), t)
        h = F.interpolate(h, size=h1.shape[2:], mode='nearest'); h = self.u4(torch.cat([h, h1], 1), t)
        return self.out(h)

class Diffusion:
    def __init__(self, timesteps, noise_type, theta0=0.001, beta_start=1e-4, beta_end=0.02):
        self.timesteps = timesteps; self.noise_type = noise_type
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1. - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, 0).to(device)
        self.sqrt_alpha_bars = torch.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1. - self.alpha_bars)
        if noise_type == 'gamma':
            self.theta_t = torch.sqrt(self.alpha_bars) * theta0
            self.k_t = self.betas / (self.alpha_bars * theta0 ** 2)
            self.k_bar_t = torch.cumsum(self.k_t, 0)
        elif noise_type != 'gaussian':
            raise ValueError(noise_type)
    def sample_timesteps(self, n):
        return torch.randint(1, self.timesteps, (n,), device=device)
    def _gamma_noise(self, shape, k_bar, theta):
        conc = k_bar.double().expand(shape).contiguous(); rate = (1.0 / theta).double().expand(shape).contiguous()
        g = torch.distributions.Gamma(conc, rate).sample()
        return g - (k_bar * theta).double().expand(shape)
    def q_sample(self, x0, t):
        sa = self.sqrt_alpha_bars[t][:, None, None, None]; so = self.sqrt_one_minus_alpha_bars[t][:, None, None, None]
        if self.noise_type == 'gaussian':
            target = torch.randn_like(x0)
        else:
            kb = self.k_bar_t[t][:, None, None, None]; th = self.theta_t[t][:, None, None, None]
            target = self._gamma_noise(x0.shape, kb, th).to(x0.dtype) / so
        return sa * x0 + so * target, target
    def p_losses(self, model, x0, y, t):
        x_t, target = self.q_sample(x0, t); pred = model(x_t, t, y)
        return F.l1_loss(target, pred) if self.noise_type == 'gamma' else F.mse_loss(target, pred)
    def _init_noise(self, shape, top):
        if self.noise_type == 'gaussian':
            return torch.randn(shape, device=device)
        return self._gamma_noise(shape, self.k_bar_t[top].view(1, 1, 1, 1),
                                 self.theta_t[top].view(1, 1, 1, 1)).to(device).float()
    @torch.no_grad()
    def ddim_sample(self, model, n, y, image_size, channels=1, steps=50):
        model.eval(); shape = (n, channels, image_size, image_size)
        seq = sorted(set(torch.linspace(0, self.timesteps - 1, steps).round().long().tolist()))[::-1]
        x = self._init_noise(shape, seq[0])
        for idx, i in enumerate(seq):
            t = torch.full((n,), i, device=device, dtype=torch.long)
            eps = model(x, t, y); ab = self.alpha_bars[i]
            ip = seq[idx + 1] if idx + 1 < len(seq) else None
            abp = self.alpha_bars[ip] if ip is not None else torch.tensor(1.0, device=device)
            x0 = ((x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)).clamp(-1, 1)
            x = torch.sqrt(abp) * x0 + torch.sqrt(1 - abp) * eps
        return x

class Classifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        def b(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.LeakyReLU(0.2))
        self.f = nn.Sequential(b(1, 32), b(32, 32), b(32, 128), b(128, 256))
        self.head = nn.Linear(256, num_classes)
    def forward(self, x): return self.head(self.f(x).flatten(1))

## 4. shared helpers

In [5]:
def metrics(y_true, pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, average='macro', zero_division=0)
    pc = precision_recall_fscore_support(y_true, pred, average=None, zero_division=0)[2]   # per-class F1
    return dict(precision=p, recall=r, fscore=f, accuracy=accuracy_score(y_true, pred),
                **{f'f1_{NAMES[i]}': pc[i] for i in range(num_classes)})

@torch.no_grad()
def predict(clf, X):
    clf.eval(); out = []
    for i in range(0, len(X), 256):
        out.append(clf(X[i:i+256].to(device)).argmax(1).cpu())
    return torch.cat(out).numpy()

def train_classifier(X, y, epochs):
    clf = Classifier(num_classes).to(device)
    opt = optim.Adam(clf.parameters(), lr=learning_rate, betas=adam_betas)
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        clf.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(clf(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return clf

@torch.no_grad()
def gen_class(diff, net, c, n):
    net.eval(); outs, rem = [], n
    while rem > 0:
        b = min(batch_size * 2, rem)
        yy = torch.full((b,), c, device=device, dtype=torch.long)
        outs.append(diff.ddim_sample(net, b, yy, image_size, steps=gen_steps).clamp(-1, 1).cpu()); rem -= b
    return torch.cat(outs)[:n]

## 5. the three experiment functions

In [6]:
import os
def _save_state(state, name):
    if save_models:
        torch.save(state, os.path.join(models_dir, name))

def run_baseline(Xtr, ytr, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed)
    clf = train_classifier(Xtr, ytr, baseline_epochs)
    _save_state(clf.state_dict(), f'{tag}_clf.pth')
    return metrics(y_test.numpy(), predict(clf, X_test))

def run_sequential(Xtr, ytr, nt, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed)
    net = UNet(num_classes).to(device); diff = Diffusion(timesteps, nt, theta0)
    opt = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=seq_diffusion_epochs)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(seq_diffusion_epochs):
        net.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device); t = diff.sample_timesteps(x.size(0))
            loss = diff.p_losses(net, x, y, t)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        sched.step()
    _save_state(net.state_dict(), f'{tag}_unet.pth')          # trained diffusion generator
    Xs, Ys = [Xtr], [ytr]
    for c in range(num_classes):
        ng = target_per_class - (ytr == c).sum().item()
        if ng > 0:
            Xs.append(gen_class(diff, net, c, ng)); Ys.append(torch.full((ng,), c, dtype=torch.long))
    clf = train_classifier(torch.cat(Xs), torch.cat(Ys), seq_classifier_epochs)
    _save_state(clf.state_dict(), f'{tag}_clf.pth')           # trained classifier
    return metrics(y_test.numpy(), predict(clf, X_test))

def run_alternating(Xtr, ytr, nt, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed)
    net = UNet(num_classes).to(device); diff = Diffusion(timesteps, nt, theta0)
    clf = Classifier(num_classes).to(device)
    opt_g = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=alt_warmup_epochs + alt_epochs)
    opt_c = optim.Adam(clf.parameters(), lr=learning_rate, betas=adam_betas)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)

    def g_step(coop):
        net.train()
        if coop:
            for p in clf.parameters(): p.requires_grad_(False)
        for x, y in loader:
            x, y = x.to(device), y.to(device); t = diff.sample_timesteps(x.size(0))
            x_t, target = diff.q_sample(x, t); pred = net(x_t, t, y)
            loss = F.l1_loss(target, pred) if diff.noise_type == 'gamma' else F.mse_loss(target, pred)
            if coop:
                sa = diff.sqrt_alpha_bars[t][:, None, None, None]; so = diff.sqrt_one_minus_alpha_bars[t][:, None, None, None]
                x0h = ((x_t - so * pred) / sa).clamp(-1, 1)
                loss = loss + alpha_coop * F.cross_entropy(clf(x0h), y)
            opt_g.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt_g.step()
        if coop:
            for p in clf.parameters(): p.requires_grad_(True)

    def c_step():
        clf.train()
        for x_real, y_real in loader:
            x_real, y_real = x_real.to(device), y_real.to(device)
            counts = torch.bincount(y_real, minlength=num_classes); m_max = int(counts.max().item())
            gx, gy = [], []
            for c in range(num_classes):
                ng = m_max - int(counts[c].item())
                if ng > 0:
                    yb = torch.full((ng,), c, device=device, dtype=torch.long)
                    gx.append(diff.ddim_sample(net, ng, yb, image_size, steps=gen_steps).clamp(-1, 1)); gy.append(yb)
            if gx: xb = torch.cat([x_real] + gx); yb = torch.cat([y_real] + gy)
            else:  xb, yb = x_real, y_real
            loss = F.cross_entropy(clf(xb), yb)
            opt_c.zero_grad(); loss.backward(); opt_c.step()

    for _ in range(alt_warmup_epochs):
        g_step(False); sched.step()
    best = None
    for ep in range(alt_epochs):
        g_step(True); sched.step(); c_step()
        if ep % eval_every == 0 or ep == alt_epochs - 1:
            m = metrics(y_test.numpy(), predict(clf, X_test))
            if best is None or m['fscore'] > best['fscore']:
                best = m
    _save_state(net.state_dict(), f'{tag}_unet.pth')          # final diffusion generator
    _save_state(clf.state_dict(), f'{tag}_clf.pth')           # final classifier
    return best

## 6. run all experiments (incremental Excel save + .pth saving)

In [7]:
def save_results(rows, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    df = pd.DataFrame(rows)
    try:
        df.to_excel(path, sheet_name='runs', index=False)
    except Exception as e:
        print('xlsx save failed, writing CSV instead:', e)
        df.to_csv(path.replace('.xlsx', '.csv'), index=False)
    return df

CONFIGS = [('sequential', 'gaussian'), ('sequential', 'gamma'),
           ('alternating', 'gaussian'), ('alternating', 'gamma')]

if save_models:
    os.makedirs(models_dir, exist_ok=True)
    print(f"saving .pth models into ./{models_dir}/")

rows = []
for r in range(n_runs):
    seed = base_seed + r
    Xtr, ytr = build_train(seed)

    m = run_baseline(Xtr, ytr, seed, tag=f'baseline_run{r}')
    rows.append(dict(run=r, seed=seed, method='baseline', noise='none', **m))
    save_results(rows, xlsx_path)
    print(f"run {r} | baseline           | F={m['fscore']:.3f}")

    for method, nt in CONFIGS:
        fn = run_sequential if method == 'sequential' else run_alternating
        m = fn(Xtr, ytr, nt, seed, tag=f'{method}_{nt}_run{r}')
        rows.append(dict(run=r, seed=seed, method=method, noise=nt, **m))
        save_results(rows, xlsx_path)
        print(f"run {r} | {method:<11} {nt:<8} | F={m['fscore']:.3f}")

results = pd.DataFrame(rows)
print(f"\nDONE. {len(results)} rows -> {xlsx_path}" + (f"; .pth files -> ./{models_dir}/" if save_models else ""))

saving .pth models into ./saved_models/
run 0 | baseline           | F=0.871
run 0 | sequential  gaussian | F=0.874
run 0 | sequential  gamma    | F=0.874
run 0 | alternating gaussian | F=0.892
run 0 | alternating gamma    | F=0.890
run 1 | baseline           | F=0.885
run 1 | sequential  gaussian | F=0.907
run 1 | sequential  gamma    | F=0.903
run 1 | alternating gaussian | F=0.875
run 1 | alternating gamma    | F=0.884
run 2 | baseline           | F=0.864
run 2 | sequential  gaussian | F=0.839
run 2 | sequential  gamma    | F=0.849
run 2 | alternating gaussian | F=0.872
run 2 | alternating gamma    | F=0.867
run 3 | baseline           | F=0.850
run 3 | sequential  gaussian | F=0.877
run 3 | sequential  gamma    | F=0.891
run 3 | alternating gaussian | F=0.876
run 3 | alternating gamma    | F=0.875
run 4 | baseline           | F=0.835
run 4 | sequential  gaussian | F=0.792
run 4 | sequential  gamma    | F=0.715
run 4 | alternating gaussian | F=0.885
run 4 | alternating gamma    | F=0

## 7. load saved Excel and show averages (standalone cell)

In [2]:
import pandas as pd

xlsx_path = 'results/experiment_results.xlsx'
saved = pd.read_excel(xlsx_path, sheet_name='runs')

metric_cols = ['precision', 'recall', 'fscore', 'accuracy']
metric_cols += [c for c in saved.columns if c.startswith('f1_')]   # per-class F1 if present

avg = saved.groupby(['method', 'noise'])[metric_cols].mean().round(4)
print(f"Averages over {saved['run'].nunique()} runs (from {xlsx_path}):\n")
print(avg)

print("\nF-score  mean ± std:")
for (method, noise), g in saved.groupby(['method', 'noise']):
    print(f"  {method:<12}{noise:<10}  {g['fscore'].mean():.4f} ± {g['fscore'].std():.4f}")

avg

Averages over 10 runs (from experiment_results.xlsx):

                      precision  recall  fscore  accuracy  f1_T-shirt  \
method      noise                                                       
alternating gamma        0.8878  0.8785  0.8791    0.8785      0.8671   
            gaussian     0.8898  0.8799  0.8806    0.8799      0.8609   
baseline    none         0.8847  0.8611  0.8624    0.8611      0.8378   
sequential  gamma        0.8859  0.8615  0.8621    0.8615      0.8419   
            gaussian     0.8866  0.8684  0.8682    0.8684      0.8504   

                      f1_Pullover  f1_Dress  
method      noise                            
alternating gamma          0.9027    0.8673  
            gaussian       0.9117    0.8692  
baseline    none           0.9073    0.8420  
sequential  gamma          0.9082    0.8361  
            gaussian       0.9155    0.8386  

F-score  mean ± std:
  alternating gamma       0.8791 ± 0.0117
  alternating gaussian    0.8806 ± 0.0126
  bas

precision  recall  fscore  accuracy  f1_T-shirt  \
method      noise                                                       
alternating gamma        0.8878  0.8785  0.8791    0.8785      0.8671   
            gaussian     0.8898  0.8799  0.8806    0.8799      0.8609   
baseline    none         0.8847  0.8611  0.8624    0.8611      0.8378   
sequential  gamma        0.8859  0.8615  0.8621    0.8615      0.8419   
            gaussian     0.8866  0.8684  0.8682    0.8684      0.8504   

                      f1_Pullover  f1_Dress  
method      noise                            
alternating gamma          0.9027    0.8673  
            gaussian       0.9117    0.8692  
baseline    none           0.9073    0.8420  
sequential  gamma          0.9082    0.8361  
            gaussian       0.9155    0.8386

## 8. summary: mean +/- std

In [8]:
summary = (results.groupby(['method', 'noise'])[['precision', 'recall', 'fscore', 'accuracy']]
           .agg(['mean', 'std']).round(4))
print(summary)

print("\nF-score  mean ± std  over", n_runs, "runs:")
for (method, noise), g in results.groupby(['method', 'noise']):
    print(f"  {method:<12}{noise:<10}  {g['fscore'].mean():.4f} ± {g['fscore'].std():.4f}")

                     precision          recall          fscore          \
                          mean     std    mean     std    mean     std   
method      noise                                                        
alternating gamma       0.8878  0.0107  0.8785  0.0116  0.8791  0.0117   
            gaussian    0.8898  0.0096  0.8799  0.0129  0.8806  0.0126   
baseline    none        0.8847  0.0089  0.8611  0.0174  0.8624  0.0172   
sequential  gamma       0.8859  0.0235  0.8615  0.0552  0.8621  0.0559   
            gaussian    0.8866  0.0216  0.8684  0.0360  0.8682  0.0377   

                     accuracy          
                         mean     std  
method      noise                      
alternating gamma      0.8785  0.0116  
            gaussian   0.8799  0.0129  
baseline    none       0.8611  0.0174  
sequential  gamma      0.8615  0.0552  
            gaussian   0.8684  0.0360  

F-score  mean ± std  over 10 runs:
  alternating gamma       0.8791 ± 0.0117
  alterna

## 9. paired t-tests

In [9]:
def paired(df, m1, n1, m2, n2, metric='fscore'):
    a = df[(df.method == m1) & (df.noise == n1)].sort_values('run')[metric].values
    b = df[(df.method == m2) & (df.noise == n2)].sort_values('run')[metric].values
    t, p2 = ttest_rel(a, b)
    p1 = p2 / 2 if t > 0 else 1 - p2 / 2          # one-sided: H1 = (first > second)
    return dict(comparison=f"{m1}/{n1}  vs  {m2}/{n2}", metric=metric,
                mean1=round(a.mean(), 4), mean2=round(b.mean(), 4), diff=round(a.mean() - b.mean(), 4),
                t_stat=round(float(t), 3), p_two_sided=round(float(p2), 4),
                p_one_sided=round(float(p1), 4), significant_5pct=bool(p1 < 0.05))

PAIRS = [
    ('sequential', 'gamma',  'sequential', 'gaussian'),   # gamma vs gaussian (sequential)
    ('alternating', 'gamma', 'alternating', 'gaussian'),  # gamma vs gaussian (alternating)
    ('sequential', 'gamma',  'baseline', 'none'),
    ('sequential', 'gaussian', 'baseline', 'none'),
    ('alternating', 'gamma', 'baseline', 'none'),
    ('alternating', 'gaussian', 'baseline', 'none'),
    ('alternating', 'gamma', 'sequential', 'gamma'),      # method effect (gamma)
    ('alternating', 'gaussian', 'sequential', 'gaussian'),# method effect (gaussian)
]
ttests = pd.DataFrame([paired(results, *p) for p in PAIRS])
pd.set_option('display.width', 200, 'display.max_columns', 20)
print("Paired t-tests on macro F-score (one-sided H1: first > second):")
print(ttests.to_string(index=False))

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as w:
    results.to_excel(w, sheet_name='runs', index=False)
    summary.to_excel(w, sheet_name='summary')
    ttests.to_excel(w, sheet_name='ttests', index=False)
print(f"\nSaved runs + summary + ttests to {xlsx_path}")

Paired t-tests on macro F-score (one-sided H1: first > second):
                                   comparison metric  mean1  mean2    diff  t_stat  p_two_sided  p_one_sided  significant_5pct
    sequential/gamma  vs  sequential/gaussian fscore 0.8621 0.8682 -0.0061  -0.719       0.4904       0.7548             False
  alternating/gamma  vs  alternating/gaussian fscore 0.8791 0.8806 -0.0016  -0.363       0.7253       0.6374             False
          sequential/gamma  vs  baseline/none fscore 0.8621 0.8624 -0.0003  -0.019       0.9855       0.5073             False
       sequential/gaussian  vs  baseline/none fscore 0.8682 0.8624  0.0058   0.696       0.5042       0.2521             False
         alternating/gamma  vs  baseline/none fscore 0.8791 0.8624  0.0167   3.478       0.0070       0.0035              True
      alternating/gaussian  vs  baseline/none fscore 0.8806 0.8624  0.0183   2.937       0.0166       0.0083              True
      alternating/gamma  vs  sequential/gamma f